
### Notebook : 01_shared_models




##### 1 : Purpose

This notebook defines the reusable shared types (Literal definitions) and shared Pydantic models (schemas) that will be used throughout the multi-agent workflow.


It contains:

- Shared type definitions
- Coordinator task schema
- Base agent-result schema
- Specialist-agent result schemas
- Execution-history schema

It does not contain:

- Shared-state creation
- Helper functions
- Tool calls
- Agent execution logic
- Tests that automatically run


##### 2 : Technologies Used

- Python
- Pydantic
- Type hints
- Literal types
- Inheritance
- UTC timestamps
- Databricks notebooks


##### 3: Input

- No customer request or tool output is processed directly.
- The notebook receives only the schema requirements of the multi-agent system.


##### 4 : Output

- Reusable Python types and Pydantic models available to all downstream notebooks.


##### 5 : Multi-Agent Architecture

The multi-agent system separates planning, specialized execution, retention
decision-making, and final response generation.

```text

Coordinator and specialist agents
              │
              ▼
      Shared Pydantic schemas
              │
      ┌───────┼───────────┐
      ▼       ▼           ▼
Execution   Agent       Execution
plans       results     records
              │
              ▼
Validated data exchanged across notebooks

```


###### 6 : Imports

In [0]:
from datetime import datetime, timezone
from typing import Any, Callable, Dict, List, Literal, Optional

from pydantic import BaseModel,  ConfigDict, Field


###### 7 : Shared type definitions

In [0]:
# ===============
# Workflow Types
# ===============


# Defines the valid specialized agents that may receive tasks from the Coordinator Agent.
AgentName = Literal[
    "coordinator_agent",
    "sql_agent",
    "prediction_agent",
    "vector_search_agent",
    "retention_agent",
    "final_response_agent",
]


# Defines every component that may create an execution-history record or workflow error.
# The orchestrator coordinates agents but does not receive Coordinator tasks, so it is not included in AgentName.

WorkflowComponent = Literal[
    "orchestrator",
    "coordinator_agent",
    "sql_agent",
    "prediction_agent",
    "vector_search_agent",
    "retention_agent",
    "final_response_agent",
]



# represents the lifecycle state of an agent or workflow step.
AgentStatus = Literal[
    "pending",
    "running",
    "success",
    "failed",
    "skipped",
]


# This represents the Coordinator Agent’s classification of the user request.
RequestType = Literal[
    "sql_analytics",
    "prediction",
    "vector_search",
    "retention",
    "combined",
    "unsupported",
]

# ====================
# Component Constants
# ====================

ORCHESTRATOR_NAME = "orchestrator"
COORDINATOR_AGENT_NAME = "coordinator_agent"
SQL_AGENT_NAME = "sql_agent"
PREDICTION_AGENT_NAME = "prediction_agent"
VECTOR_SEARCH_AGENT_NAME = "vector_search_agent"
RETENTION_AGENT_NAME = "retention_agent"
FINAL_RESPONSE_AGENT_NAME =  "final_response_agent"


# ===============
# Business Types
# ===============

# This is a business-domain type.It represents valid customer-support classifications
SupportCategory = Literal[
    "Billing",
    "Technical",
    "Service",
    "Account",
    "General",
]


#This defines the allowed outputs from the Retention Tool and Retention Agent.
RetentionAction = Literal[
    "offer_discount",
    "offer_support_package",
    "service_quality_review",
    "billing_review",
    "no_action",
]

# ==========================
# Shared business constants
# ==========================


RETENTION_ACTION_DESCRIPTIONS: Dict[
    RetentionAction,
    str,
] = {
    "offer_discount": (
        "Offer a targeted pricing discount when cost, "
        "pricing, or affordability is the primary "
        "retention concern."
    ),

    "offer_support_package": (
        "Provide proactive or enhanced customer support "
        "when the customer may benefit from additional "
        "guidance, follow-up, or technical assistance."
    ),

    "service_quality_review": (
        "Initiate a review of service reliability, "
        "performance, outages, speed, or other "
        "service-quality concerns."
    ),

    "billing_review": (
        "Review invoices, charges, payment issues, or "
        "billing disputes and correct any identified "
        "problems."
    ),

    "no_action": (
        "Do not recommend a retention intervention when "
        "the customer is not currently considered at "
        "meaningful churn risk."
    ),
}


# ========================
# Tool function signatures
# ========================

SQLAnalyticsFunction = Callable[
    [str],
    Dict[str, Any],
]

PredictionToolFunction = Callable[
    [str],
    Dict[str, Any],
]

VectorSearchToolFunction = Callable[
    [str, int],
    Dict[str, Any],
]

RetentionToolFunction = Callable[
    [Dict[str, Any]],
    Dict[str, Any],
]

LLMInvokeFunction = Callable[
    [str],
    str,
]

This contract means that any injected Retention Tool must:

- Accept:  Dict[str, Any]
- Return:  Dict[str, Any]

The tool implementation can later be replaced with:

- deterministic business rules,
- an LLM-backed tool,
- an external service,
- or a hybrid implementation.

The Retention Agent does not need to change as long as the new tool follows the same contract.


##### 8 : Shared Pydantic Schemas

In [0]:
# 1 : Coordinator task schema

class AgentTask(BaseModel):
    """
    Represents one task in the Coordinator Agent's execution plan.
    """

    task_id: str

    agent_name: AgentName

    task_description: str

    depends_on: List[AgentName] = Field(
        default_factory=list
    )

In [0]:
# 2 : Base agent-result schema

class BaseAgentResult(BaseModel):
    """
    Common fields returned by every agent.
    """

    agent_name: AgentName

    status: AgentStatus

    message: str

    task_description: Optional[str] = None

    error: Optional[str] = None

In [0]:
# 3 : Coordinator result schema

class CoordinatorResult(BaseAgentResult):
    """
    Validated result returned by the Coordinator Agent.
    """

    agent_name: Literal["coordinator_agent"] = (
    "coordinator_agent"
    )

    request_type: RequestType

    reasoning: str

    execution_plan: List[AgentTask] = Field(
        default_factory=list
    )

In [0]:
# 4 : SQL Agent result schema

class SQLAgentResult(BaseAgentResult):
    """
    Validated result returned by the SQL Agent.
    """

    agent_name: Literal["sql_agent"] = "sql_agent"

    sql_action: Optional[str] = None

    sql_result: Any = None

    row_count: Optional[int] = None

# The sql_result field uses Any because the SQL tool may return: Spark Row objects, Lists of rows, Dictionaries, Numeric aggregates, Tabular results

In [0]:
# 5 : Prediction Agent result schema

ChurnPredictionCategory = Literal[
    "Churn",
    "No Churn",
]


class PredictionAgentResult(BaseAgentResult):
    """
    Validated result returned by the Prediction Agent.
    """

    agent_name: Literal["prediction_agent"] = (
        "prediction_agent"
    )

    customer_id: Optional[str] = None

    predicted_category: Optional[
        ChurnPredictionCategory
    ] = None

    confidence: Optional[float] = Field(
        default=None,
        ge=0.0,
        le=1.0,
    )

    model_name: Optional[str] = None

    raw_prediction: Any = None

# The raw_prediction field preserves the original Prediction Tool output when needed.

In [0]:
class VectorSearchItem(BaseModel):
    """
    One customer-note result returned by semantic search.
    """

    customer_id: str

    note: str

    similarity_score: Optional[float] = Field(
        default=None,
        ge=0.0,
        le=1.0,
    )

In [0]:
# 6 : Vector Search Agent result schema

class VectorSearchAgentResult(BaseAgentResult):
    """
    Validated output returned by the Vector Search Agent.
    """

    agent_name: Literal["vector_search_agent"] = (
    "vector_search_agent"
    )

    task_id: Optional[str] = None

    query: Optional[str] = None

    results: List[VectorSearchItem] = Field(
        default_factory=list
    )

In [0]:
# 7 : Retention Agent result schema

class RetentionAgentResult(BaseAgentResult):
    """
    Validated output returned by the Retention Agent.
    
    """

    agent_name: Literal["retention_agent"] = (
        "retention_agent"
    )

    task_id: Optional[str] = None

    customer_id: Optional[str] = None

    recommended_action: Optional[
        RetentionAction
    ] = None

    action_reason: Optional[str] = None

    prediction_label: Optional[
        ChurnPredictionCategory
    ] = None

    prediction_confidence: Optional[float] = Field(
        default=None,
        ge=0.0,
        le=1.0,
    )

    supporting_notes: List[str] = Field(
        default_factory=list
    )

In [0]:
# 8 : Final Response Agent result schema

class FinalResponseAgentResult(BaseAgentResult):
    """
    Validated result returned by the Final Response Agent.
    """

    agent_name: Literal["final_response_agent"] = (
        "final_response_agent"
    )

    final_response: Optional[str] = None

In [0]:
# 9 : Agent Error Record schema

class AgentErrorRecord(BaseModel):
    """
    Represents one agent or workflow-component error.
    """

    agent_name: WorkflowComponent

    error_code: str

    error_message: str

    timestamp: datetime = Field(
        default_factory=lambda: datetime.now(
            timezone.utc
        )
    )

##### 9 : ExecutionRecord

In [0]:
# Execution-history schema: An ExecutionRecord represents one event in the workflow execution history.

class ExecutionRecord(BaseModel):
    """
    Represents one agent execution event in the multi-agent workflow.
    """

    agent_name: WorkflowComponent

    status: AgentStatus

    message: str

    timestamp: datetime = Field(
        default_factory=lambda: datetime.now(timezone.utc)
    )

##### 10 : Schema Validation


In [0]:
def validate_shared_schemas() -> None:
    """
    Run basic validation checks for shared schemas.

    Call manually only when testing this notebook.
    """

    example_task = AgentTask(
        task_id="task_1",
        agent_name=PREDICTION_AGENT_NAME,
        task_description=(
            "Predict customer churn."
        ),
    )

    example_prediction_result = (
        PredictionAgentResult(
            status="success",
            message=(
                "Prediction completed successfully."
            ),
            predicted_category="Churn",
            confidence=0.91,
        )
    )

    example_agent_execution_record = (
        ExecutionRecord(
            agent_name=PREDICTION_AGENT_NAME,
            status="success",
            message=(
                "Prediction task completed successfully."
            ),
        )
    )

    example_orchestrator_record = (
        ExecutionRecord(
            agent_name=ORCHESTRATOR_NAME,
            status="running",
            message=(
                "The orchestrator started the workflow."
            ),
        )
    )

    example_orchestrator_error = (
        AgentErrorRecord(
            agent_name=ORCHESTRATOR_NAME,
            error_code="TEST_WORKFLOW_ERROR",
            error_message=(
                "Test workflow error."
            ),
        )
    )

    assert (
        example_task.agent_name
        == PREDICTION_AGENT_NAME
    )

    assert example_task.depends_on == []

    assert (
        example_prediction_result.agent_name
        == PREDICTION_AGENT_NAME
    )

    assert (
        example_prediction_result.predicted_category
        == "Churn"
    )

    assert (
        example_prediction_result.confidence
        == 0.91
    )

    assert (
        example_agent_execution_record.agent_name
        == PREDICTION_AGENT_NAME
    )

    assert (
        example_agent_execution_record.status
        == "success"
    )

    assert (
        example_orchestrator_record.agent_name
        == ORCHESTRATOR_NAME
    )

    assert (
        example_orchestrator_record.status
        == "running"
    )

    assert (
        example_orchestrator_error.agent_name
        == ORCHESTRATOR_NAME
    )

    assert (
        example_orchestrator_error.error_code
        == "TEST_WORKFLOW_ERROR"
    )

    assert (
        example_agent_execution_record.timestamp
        is not None
    )

    assert (
        example_orchestrator_error.timestamp
        is not None
    )

    print(
        "All shared-schema validation checks passed."
    )


##### 11 : Key learnings

• A multi-agent system divides a complex workflow among specialized agents.

• Each agent should have one clearly defined responsibility.

• Shared Literal types restrict workflow and business values to approved options.

• Pydantic models create explicit communication contracts between agents.

• Schema inheritance avoids repeating common agent-result fields.

• Agent dependencies describe the required execution order.

• Pydantic validates agent outputs before downstream components use them.

• Structured agent results should remain separate from the final customer-facing response.

• Consistent agent names, statuses, messages, and errors make workflows  easier to test and debug.

• Timezone-aware UTC timestamps provide consistent execution records.



##### 12 : Conclusion

This notebook established the shared data contracts for the multi-agent customer-support system.

The architecture now contains:

- Standardized agent names, statuses, and request types
- Valid support categories and retention actions
- Dependency-aware Coordinator tasks
- A reusable base result model
- Validated Coordinator and specialist-agent result schemas
- A validated execution-history record schema

These shared types and Pydantic models ensure that all downstream notebooks exchange information using consistent and validated structures. Shared-state creation and reusable state-management functions will be implemented in the next notebook.


##### 13 : Next Notebook

Next notebook defines the shared workflow state and reusable state-management functions.